CSU Transparent California


Make sure to download the transparent california zip from the Raw files folder in Google Drive. Upload all csv's from the CSU folder in the zip.

In [20]:
# Lossless data processing for provided downloaded from Transparent Califronia

import polars as pl

print("Scanning CSV files...")

# Numerical cols
financial_cols = [
    "base", "overtime", "other", "totalpay", "totalbenefits",
    "totalpaybenefits"
]

df = pl.scan_csv(
    "content/tc_raw/**/*.csv",
    include_file_paths="filepath"
)
# Empty Cols
df = df.drop(['ual', 'benefitsee', 'benefitser', 'benefitsdc'], strict=False)

# Helper function to currency text into Ints!
def clean_currency(col_name):
    return ( pl.col(col_name) .cast(pl.Int32, strict=False))

df = df.rename(str.lower)

# Apply typing and data transformations
df = df.with_columns(
    pl.selectors.string().str.to_lowercase()
).with_columns(
    # Get Year From Filename
    pl.col("filepath").str.extract(r"-(\d{4})\.csv$", 1).cast(pl.Int16).alias("year"),

    # Categorical cols
    pl.col("job").cast(pl.Categorical),
    pl.col("department").cast(pl.Categorical),

    # Clean and cast all financial columns
    *[clean_currency(col) for col in financial_cols]
)

df = df.drop(["filepath"])

final_df = df.collect()
final_df.write_parquet("content/csu_salaries_2013-2024.parquet", compression="zstd")

print(f"Processed {final_df.height:,} records.")

Scanning CSV files...
Processed 987,543 records.


In [21]:
final_df.describe()

statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status
str,str,str,str,f64,f64,f64,str,f64,f64,f64,str,f64,str,str
"""count""","""987543""","""987543""","""0""",987543.0,987543.0,987543.0,"""0""",987543.0,987543.0,987543.0,"""28458""",987543.0,"""987543""","""0"""
"""null_count""","""0""","""0""","""987543""",0.0,0.0,0.0,"""987543""",0.0,0.0,0.0,"""959085""",0.0,"""0""","""987543"""
"""mean""",null,null,null,38954.754078,251.729386,2990.511412,null,42197.157741,18286.612329,60484.049206,null,2019.001874,null,null
"""std""",null,null,null,41626.826402,2194.793662,7709.038972,null,44443.447239,20332.057299,63379.724403,null,3.35553,null,null
"""min""",""" judiyaba""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null
"""25%""",null,null,null,412.0,0.0,0.0,null,2430.0,0.0,2500.0,null,2016.0,null,null
"""50%""",null,null,null,29019.0,0.0,2.0,null,32434.0,11413.0,46481.0,null,2019.0,null,null
"""75%""",null,null,null,63965.0,0.0,2500.0,null,67690.0,33840.0,100952.0,null,2022.0,null,null
"""max""","""zvonimir hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null


Transparent California Filtered CSU Data

In [25]:
import re

import pandas as pd

CSU_ROLES = [
    ("supplemental_teaching_pay", r"special programs|summer session|extension|additional employment|music studio"),
    ("ladder_faculty", r"^instructional faculty|^grant-related/specially funded instructional faculty|^department chair|^cma instructional faculty"),
    ("lecturer", r"lecturer"),
    ("clinical_adjunct_faculty", r"^visiting faculty"),
    ("postdoc", r"^research fellow|post-doctoral"),
    ("trainee", r"counselor intern"),
    ("researcher_specialist_librarian", r"librarian|academic-related|coach"),
]


def role_of(title, rules) -> str:
    t = str(title or "").lower().strip()
    return next((role for role, pattern in rules if re.search(pattern, t)), "staff")


csu_salaries = pd.read_parquet("content/csu_salaries_2013-2024.parquet")

# Make sure pay is numeric before filtering
csu_salaries["base"] = pd.to_numeric(csu_salaries["base"], errors="coerce")

filtered_csu_salaries = csu_salaries.copy()
filtered_csu_salaries["role"] = filtered_csu_salaries["job"].map(lambda t: role_of(t, CSU_ROLES)).astype("string")

# Replace "name" with your actual name-column name if needed
filtered_csu_salaries["name"] = (
    filtered_csu_salaries["name"]
    .str.title()
    .str.replace(r"(?<=\s)[A-Z]\.?(?=\s|$)", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

filtered_csu_salaries.to_parquet(
    "content/transparent_california_filtered_csu_salaries_2013-2024.parquet",
    index=False
)

better_df = pl.from_pandas(filtered_csu_salaries)
better_df.describe()

statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status,role
str,str,str,f64,f64,f64,f64,str,f64,f64,f64,str,f64,str,str,str
"""count""","""987543""","""987543""",0.0,987543.0,987543.0,987543.0,"""0""",987543.0,987543.0,987543.0,"""28458""",987543.0,"""987543""","""0""","""987543"""
"""null_count""","""0""","""0""",987543.0,0.0,0.0,0.0,"""987543""",0.0,0.0,0.0,"""959085""",0.0,"""0""","""987543""","""0"""
"""mean""",null,null,null,38954.754078,251.729386,2990.511412,null,42197.157741,18286.612329,60484.049206,null,2019.001874,null,null,null
"""std""",null,null,null,41626.826402,2194.793662,7709.038972,null,44443.447239,20332.057299,63379.724403,null,3.35553,null,null,null
"""min""","""A Alisia Nantze""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null,"""clinical_adjunct_faculty"""
"""25%""",null,null,null,412.0,0.0,0.0,null,2430.0,0.0,2500.0,null,2016.0,null,null,null
"""50%""",null,null,null,29019.0,0.0,2.0,null,32434.0,11413.0,46481.0,null,2019.0,null,null,null
"""75%""",null,null,null,63965.0,0.0,2500.0,null,67690.0,33840.0,100952.0,null,2022.0,null,null,null
"""max""","""Zvonimir Hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null,"""trainee"""


No Duplicates in each year for each role

In [26]:
csu_salaries = pd.read_parquet("content/transparent_california_filtered_csu_salaries_2013-2024.parquet")

no_dup_filtered_csu_salaries = csu_salaries.drop_duplicates(subset=["year", "name", "role"])

no_dup_filtered_csu_salaries.to_parquet(
    "content/transparent_california_no_dup_filtered_csu_salaries_2013-2024.parquet",
    index=False
)

better_df = pl.from_pandas(no_dup_filtered_csu_salaries)
better_df.describe()

statistic,name,job,department,base,overtime,other,benefitshw,totalpay,totalbenefits,totalpaybenefits,notes,year,agency,status,role
str,str,str,str,f64,f64,f64,str,f64,f64,f64,str,f64,str,str,str
"""count""","""896675""","""896675""","""0""",896675.0,896675.0,896675.0,"""0""",896675.0,896675.0,896675.0,"""28261""",896675.0,"""896675""","""0""","""896675"""
"""null_count""","""0""","""0""","""896675""",0.0,0.0,0.0,"""896675""",0.0,0.0,0.0,"""868414""",0.0,"""0""","""896675""","""0"""
"""mean""",null,null,null,41222.267117,267.339624,3124.777995,null,44614.555987,19342.963707,63957.815901,null,2018.84158,null,null,null
"""std""",null,null,null,41535.705278,2256.237955,7815.988307,null,44213.608331,20354.503692,63091.091877,null,3.379371,null,null,null
"""min""","""A Alisia Nantze""",null,null,-5669.0,-72.0,-23732.0,null,0.0,-95066.0,-64657.0,"""employee held multiple positio…",2013.0,"""california state university""",null,"""clinical_adjunct_faculty"""
"""25%""",null,null,null,3304.0,0.0,0.0,null,5078.0,0.0,5376.0,null,2016.0,null,null,null
"""50%""",null,null,null,33460.0,0.0,90.0,null,36606.0,15516.0,54057.0,null,2019.0,null,null,null
"""75%""",null,null,null,66250.0,0.0,2818.0,null,70157.0,34763.0,104234.0,null,2022.0,null,null,null
"""max""","""Zvonimir Hlousek""",null,null,790125.0,249767.0,473046.0,null,970004.0,181837.0,1.114304e6,"""employee holds multiple positi…",2024.0,"""california state university""",null,"""trainee"""
